# P=QK^T

### copy排列
> cp.async.cg,ldmatrix,128bit load/store：单指令最多16字节 = 128bit
> vec:每条硬件指令搬的元素数

- >bf16(2B): vec<= 8
- >每线程恰好搬一个vec向量
- >让线程列覆盖D数据
- >维度D + vec 算出线程列数
- >线程数量约束算出线程行数
- >tile行数 + 线程行数算出线程行(m)方向重复次数

### mma

>gemm m16n8k16 覆盖1warp(32thread) (m=16,n=8,k=16): [16,8] = [16,k=16]*[16,8]+[16,8]

- >thread_nums->warp_nums
1. #### tiled_mma
    >out:layout设计

        ```
        例子:warps = nt / 32
        layout = (WM,WN,WK) = 保证P一行在同一warp方便softmax = (warps,1,1)
        ```
2. #### thr_tile:线程tile视图
    >thr_ = tiled_mma.get_slice(tidx)
3. #### thr_.partition_ABC(smem):ABC smem线程视图（无数据）
    >out:tCsA:[(m0,m1,m2),mma_m,mma_n] = [(smem数据排列),m方向重复数，n方向重复数]

        ```
        例子：tile=QKV:[Bq,D]
        m16n8k16
        (m,n) += (m,k) *(k,n)
        mma_atom:C[16,8] A[16*16] B[16*8]
        mma_m = tile_m / (WM*16) 
        mma_n = tile_n / (WN*8)
        mma_k = tile_k / (WK)
        ```
4. #### tiled_mma.make_fragment_ABC(tCsA):寄存器视图（无数据）
    > out:tCrA:[(m0,m1,m2),mma_m,mma_n] = [(register数据排列),m方向重复数，n方向重复数]

5. 加载数据(见ldmatrix)+初始化```acc = cute.make_rmem_tensor(thr_mma.partition_shape_C((BQ_, BQ_)), cutlass.Float32)
        acc.fill(0.0)```

6. ```cute.gemm(tiled_mma, acc, tCrA[None, None, k], tCrB[None, None, k], acc)  ```

### ldmatrix(copy s2r)
1. copy atom:
        - x4：一条指令读 4 个 8×8 子矩阵
    ```
    atom_s2r_A = cute.make_copy_atom(
                cute.nvgpu.warp.LdMatrix8x8x16bOp(transpose=False, num_matrices=4), Q.element_type)
    ```

2. make_tiled_copy_AB:根据mma_tile生成对应copy tile
        `tiled_s2r_A = cute.make_tiled_copy_A(atom_s2r_A, tiled_mma)`

3. 线程视图:`thr_s2r_AB = tiled_s2r_AB.get_slice(tidx)`

4. ldmatrix视角源:``` tAsA = thr_s2r_A.partition_S(sQ) ```

5. ldmatrix视角寄存器,用retile重排mma视角寄存器即可: ```tArA = thr_s2r_A.retile(tCrA)  ```

6. copy执行：```cute.copy(tiled_s2r_A, tAsA[None, None, k], tArA[None, None, k])  ```

In [ ]:
import torch
import cutlass.cute as cute
import cutlass

D = 64           # 特征维度            
BQ = 64 # tile-行
HAS_CUDA = torch.cuda.is_available()


class QK_GEMM:

    def __init__(self,
                use_pred: bool = True,
                zero_fill: bool = True, 
                debug: bool = False,
                num_threads: int = 256):
        self.use_pred, self.zero_fill, self.debug = use_pred, zero_fill, debug

        # copy g2s layout
        self.NT = num_threads
        self.BQ, self.D, self.VEC = BQ, D, 8              # 8 个 bf16 = 16B = 128 位(vec-每条硬件指令搬的元素数)
        self.thr_n = D // self.VEC                        # 每行线程
        self.thr_m = num_threads // self.thr_n            # 行数
        assert BQ % self.thr_m == 0, "Bq 必须是线程行数的整数倍" # 每tile处理BQ行->重复BQ//thr_m次



    @cute.kernel
    def kernel(self, 
                Q: cute.Tensor, 
                K: cute.Tensor, 
                P: cute.Tensor, 
                tiled_copy_g2s: cute.TiledCopy,
                # tiled_copy_s2r: cute.TiledCopy,
                tiled_mma: cute.TiledMma,
                ):

        tidx, _, _ = cute.arch.thread_idx()
        bidx, bidy, _ = cute.arch.block_idx()             # bidx: 第几个 Q tile；bidy: 第几个 batch
        BQ_, D_, VEC = self.BQ, self.D, self.VEC
        smem = cutlass.utils.SmemAllocator() # The class "SmemAllocator" is deprecated
                                            # cutlass.utils.smem_allocator.SmemAllocator is deprecated; use cutlass.memory.SmemAllocator instead. It will be removed in a future release.
        sQ = smem.allocate_tensor(Q.element_type, cute.make_layout((BQ_, D_), stride=(D_, 1)), 16)
        sK = smem.allocate_tensor(K.element_type, cute.make_layout((BQ_, D_), stride=(D_, 1)), 16)

        # 物理指针
        gsrcQ = Q[(bidy, None, None)]                   # (N, D)
        gsrcK = K[(bidy, None, None)]                   # (N, D)
        gsrcP = P[(bidy, None, None)]                   #      (Npad, D)
        N = gQ.shape[0]                                 # 动态

        #===========================g2s=======================
        # block视图
        gQ = cute.local_tile(gsrcQ, (BQ_, D_), (bidx, 0))          # (Bq, D)，最后一个 tile 可能越界
        gK = cute.local_tile(gsrcK, (BQ_, D_), (bidx, 0))
        gP = cute.local_tile(gsrcP, (BQ_, D_), (bidx, 0))


        thr = tiled_copy_g2s.get_slice(tidx)
        tQgQ = thr.partition_S(gQ)                      # 每线程 gmem 视图
        tQsQ = thr.partition_D(sQ)                     # 每线程 smem 视图
        tKgK = thr.partition_S(gK)                      
        tKsK = thr.partition_D(sK)                      

        if self.debug:                                    # Python 静态分支：trace 时打印 layout
            print("tQgQ:", tQgQ)                          
            print("tQsQ:", tQsQ)


        n_rep = tQgQ.shape[0][1]                          # 一条指令内的向量重复数（静态，=1）
        n_m = cute.size(tQgQ, mode=[1])                   # 沿行方向的重复数（静态，=Bq/32=2）
        n_k = cute.size(tQgQ, mode=[2])                   # 沿列方向的重复数（静态，=1）

        # copy-pred
        if self.use_pred:
            # pred 是"每个 16B 向量一个"，形状 (n_rep, n_m, n_k)，不含向量内的 8
            pred = cute.make_rmem_tensor((n_rep, n_m, n_k), cutlass.Boolean)    
            for rv in range(n_rep):                       # 普通 Python range：编译期展开
                for m in range(n_m):
                    for k in range(n_k):
                        row = tQgQ[(0, rv), m, k][0]      # 这个向量的行号（取向量内第 0 个元素的坐标）
                        pred[rv, m, k] = row < N
            cute.copy(tiled_copy_g2s, tQgQ, tQgQ, pred=pred)
            cute.copy(tiled_copy_g2s, tKgK, tKsK, pred=pred)
        else:
            cute.copy(tiled_copy_g2s, tQgQ, tQsQ)
            cute.copy(tiled_copy_g2s, tKgK, tKsK)

        cute.arch.cp_async_commit_group()
        cute.arch.cp_async_wait_group(0)
        cute.arch.barrier()                               # 本测试里每个线程只读自己搬的数据，其实不需要；
                                                          # 留着是为了养成习惯：真 kernel 里别的线程要读

        # ---- 越界行补零（K/V 的越界行必须是 0，否则 P·V 里 0×NaN = NaN）----
        if self.zero_fill:
            zero = cutlass.BFloat16(0.0)                  # [A4]
            for rv in range(n_rep):
                for m in range(n_m):
                    for k in range(n_k):
                        row = tQgQ[(0, rv), m, k][0]
                        if row >= N:
                            for e in range(VEC):          # VEC 是 Python int → 静态展开
                                tQgQ[(e, rv), m, k] = zero
                                tKgK[(e, rv), m, k] = zero


        #===========================s2r=======================
        tiled_copy_A = cute.make_tiled_copy_A(cute.make_copy_atom(cute.nvgpu.warp.LdMatrix8x8x16bOp(),tiled_mma))
        tiled_copy_B = cute.make_tiled_copy_B(cute.make_copy_atom(cute.nvgpu.warp.LdMatrix8x8x16bOp(),tiled_mma))
        tAsA = tiled_copy_A.Partition_A(sQ)
        tBsB = tiled_copy_B.Partition_B(sK)

        #===========================gemm======================

        #===========================r2g=======================
        for i in range(cute.size(tPsP)):                  # 线性下标，两个视图同形状，colex 顺序一致
            gP[i] = tSsS[i]

    @cute.jit
    def __call__(self, 
                Q: cute.Tensor,
                K: cute.Tensor,
                P: cute.Tensor, 
                ):
        assert Q.element_type == cutlass.BFloat16
        B = Q.shape[0]
        Nq = Q.shape[1]
        Nk = K.shape[1]
        Np = P.shape[1]
        # 用 P 的静态常数重建 layout：行 stride = D（静态），这样编译器能证明每行起点 16B 对齐
        src3_Q = cute.make_tensor(Q.iterator, cute.make_layout((B, Nq, self.D), stride=(Nq * self.D, self.D, 1)))
        src3_K = cute.make_tensor(Q.iterator, cute.make_layout((B, Nk, self.D), stride=(Nk * self.D, self.D, 1)))
        dst3_P = cute.make_tensor(P.iterator, cute.make_layout((B, Np, self.D), stride=(Np * self.D, self.D, 1)))

        # atom
        copy_atom = cute.make_copy_atom(cute.nvgpu.cpasync.CopyG2SOp(), Q.element_type, num_bits_per_copy=128)
        mma_atom = cute.make_mma_atom(
            cute.nvgpu.warp.MmaF16BF16Op(cutlass.BFloat16,cutlass.Float32,(16,8,16)),
        )

        # tiled_copy
        tiled_copy_g2s = cute.make_tiled_copy_tv(copy_atom,
                                                cute.make_layout((self.thr_m, self.thr_n), stride=(self.thr_n, 1)),   # 线程：32 行 × 8 列，相邻线程沿 d 连续 → 合并访问
                                                cute.make_layout((1, self.VEC)),                                      # 每线程：1 行 × 8 个元素
                                                )
        # tiled_mma
        # warps = nt / 32
        # layout = (WM,WN,WK) = 保证P一行在同一warp方便softmax = (warps,1,1)
        # partition_ABC:
        # tile=QKV:[Bq,D] = [64,64]
        # m16n8k16
        # (m,n) += (m,k) *(k,n)
        # mma_atom:C[16,8] A[16*16] B[16*8]
        # mma_m = tile_m / (WM*16) 
        # mma_n = tile_n / (WN*8)
        # mma_k = tile_k / (WK)
        
        
        tiled_mma = cute.make_tiled_mma(mma_atom,
                                        cute.make_alyout((self.NT//32,1,1)),
                                        )
        

        
        self.kernel(src3_Q,
                    src3_K, 
                    dst3_P, 
                    tiled_copy_g2s,
                    #tiled_copy_s2r,
                    tiled_mma).launch(
            grid=((Nq + self.BQ - 1) // self.BQ, B, 1),
            block=(self.NT, 1, 1),
            smem=self.BQ * self.D * 2,
        )

### tile:
[Bq,D]*[Bk,D]^T = [Bq,Bk]

Bq = Bk

我们一般要求[Bq,N]:Bk为内部循环，此处省略。

In [4]:
import torch
import cutlass
import cutlass.cute as cute

D = 64    # 特征维度
BQ = 64   # tile 行数（本测试里 K tile 行数也是 BQ，且 BQ == D，使 S 恰好存进 (Np, D) 的 P）
HAS_CUDA = torch.cuda.is_available()


class QK_GEMM:
    """每个 block 算一个 tile：P_tile(BQ×BQ) = Q_tile(BQ×D) · K_tile(BQ×D)^T"""

    def __init__(self, use_pred: bool = True, zero_fill: bool = True,
                 debug: bool = False, num_threads: int = 128):
        self.use_pred, self.zero_fill, self.debug = use_pred, zero_fill, debug
        self.BQ, self.D, self.VEC = BQ, D, 8   # 8 个 bf16 = 16B = 128 bit，cp.async 单条搬运量

        # ---- warp 划分：atom_layout = (WM, 1, 1) ----
        # 每个 warp 至少占 M 方向 16 行，所以 WM = BQ / 16。
        # 不切 N：让每个 warp 持有完整的 S 行，将来 softmax 的行规约不跨 warp。
        # 不切 K：避免跨 warp 归约部分和。
        self.WM = BQ // 16
        self.NT = num_threads
        assert self.NT == self.WM * 32, f"BQ={BQ} 需要恰好 {self.WM*32} 个线程"

        # ---- g2s 拷贝的线程排布 ----
        self.thr_n = D // self.VEC              # 每行 8 个线程，每线程 1 个 16B 向量
        self.thr_m = self.NT // self.thr_n      # 一轮覆盖的行数（128 线程 → 16 行）
        assert BQ % self.thr_m == 0, "BQ 必须是线程行数的整数倍"   # 重复 BQ/thr_m 次

    @cute.kernel
    def kernel(self, Q: cute.Tensor, K: cute.Tensor, P: cute.Tensor,
               tiled_copy_g2s: cute.TiledCopy, tiled_mma: cute.TiledMma):

        tidx, _, _ = cute.arch.thread_idx()
        bidx, bidy, _ = cute.arch.block_idx()   # bidx: 第几个 tile；bidy: 第几个 batch
        BQ_, D_, VEC = self.BQ, self.D, self.VEC

        # ================= smem =================
        # 注意：无 swizzle 时行跨度 128B，ldmatrix 会有 bank conflict。
        # 先保证正确性；之后可给 layout 套 swizzle 做优化。
        smem = cutlass.utils.SmemAllocator()    # 警告建议改用 cutlass.memory.SmemAllocator
        sQ = smem.allocate_tensor(Q.element_type, cute.make_layout((BQ_, D_), stride=(D_, 1)), 16)
        sK = smem.allocate_tensor(K.element_type, cute.make_layout((BQ_, D_), stride=(D_, 1)), 16)

        # ================= 选出本 batch =================
        gsrcQ = Q[(bidy, None, None)]           # (Nq, D)
        gsrcK = K[(bidy, None, None)]           # (Nk, D)
        gsrcP = P[(bidy, None, None)]           # (Np, D)，Np 已 pad 成 BQ 的倍数
        Nq = gsrcQ.shape[0]                     # 动态：真实行数
        Nk = gsrcK.shape[0]

        # ================= g2s：block 视图 =================
        gQ = cute.local_tile(gsrcQ, (BQ_, D_), (bidx, 0))   # (BQ, D)，最后一块可能越界
        gK = cute.local_tile(gsrcK, (BQ_, D_), (bidx, 0))
        gP = cute.local_tile(gsrcP, (BQ_, D_), (bidx, 0))

        # identity tensor：内容就是"坐标"。与数据张量做同样的 tile/partition，
        # 就能拿到每个元素的【全局行号】，用来做越界判断。
        cQ = cute.local_tile(cute.make_identity_tensor(gsrcQ.shape), (BQ_, D_), (bidx, 0))
        cK = cute.local_tile(cute.make_identity_tensor(gsrcK.shape), (BQ_, D_), (bidx, 0))

        thr = tiled_copy_g2s.get_slice(tidx)
        tQgQ = thr.partition_S(gQ)     # 每线程 gmem 视图  ((VEC, rep), m, k)
        tQsQ = thr.partition_D(sQ)     # 每线程 smem 视图，同形状
        tKgK = thr.partition_S(gK)
        tKsK = thr.partition_D(sK)
        tQcQ = thr.partition_S(cQ)     # 每线程"坐标"视图
        tKcK = thr.partition_S(cK)

        if self.debug:                 # Python 静态分支：trace 时打印 layout
            print("tQgQ:", tQgQ)
            print("tQsQ:", tQsQ)

        n_rep = tQgQ.shape[0][1]       # 一条指令内的向量重复数
        n_m = cute.size(tQgQ, mode=[1])
        n_k = cute.size(tQgQ, mode=[2])

        if self.use_pred:
            # pred：每个 16B 向量一个布尔，形状 (n_rep, n_m, n_k)，不含向量内的 VEC
            predQ = cute.make_rmem_tensor((n_rep, n_m, n_k), cutlass.Boolean)
            predK = cute.make_rmem_tensor((n_rep, n_m, n_k), cutlass.Boolean)
            for rv in cutlass.range_constexpr(n_rep):
                for m in cutlass.range_constexpr(n_m):
                    for k in cutlass.range_constexpr(n_k):
                        # 取向量内第 0 个元素的坐标，[0] 是全局行号
                        predQ[rv, m, k] = tQcQ[(0, rv), m, k][0] < Nq
                        predK[rv, m, k] = tKcK[(0, rv), m, k][0] < Nk
            cute.copy(tiled_copy_g2s, tQgQ, tQsQ, pred=predQ)   # 目的地是 smem
            cute.copy(tiled_copy_g2s, tKgK, tKsK, pred=predK)
        else:
            cute.copy(tiled_copy_g2s, tQgQ, tQsQ)
            cute.copy(tiled_copy_g2s, tKgK, tKsK)

        cute.arch.cp_async_commit_group()
        cute.arch.cp_async_wait_group(0)
        cute.arch.barrier()

        # ---- 越界行补零（写 smem，不是 gmem）----
        # pred=False 时 cp.async 不写 smem，里面是残留垃圾；K/V 的垃圾行必须清零，
        # 否则后续 P·V 会出现 0×NaN。
        if self.zero_fill:
            zero = cutlass.BFloat16(0.0)
            for rv in cutlass.range_constexpr(n_rep):
                for m in cutlass.range_constexpr(n_m):
                    for k in cutlass.range_constexpr(n_k):
                        if tQcQ[(0, rv), m, k][0] >= Nq:
                            for e in cutlass.range_constexpr(VEC):
                                tQsQ[(e, rv), m, k] = zero
                        if tKcK[(0, rv), m, k][0] >= Nk:
                            for e in cutlass.range_constexpr(VEC):
                                tKsK[(e, rv), m, k] = zero
            # 补零是普通 st.shared，之后别的线程要用 ldmatrix 读 → 必须再同步一次
            cute.arch.barrier()

        # ================= MMA 侧的 fragment =================
        thr_mma = tiled_mma.get_slice(tidx)
        tCsA = thr_mma.partition_A(sQ)     # ((2,2,2), MMA_M, MMA_K)，smem 视图（仅用于定形状）
        tCsB = thr_mma.partition_B(sK)     # ((2,2),   MMA_N, MMA_K)
        tCrA = tiled_mma.make_fragment_A(tCsA)   # 同形状的寄存器 fragment
        tCrB = tiled_mma.make_fragment_B(tCsB)
        tCgC = thr_mma.partition_C(gP)     # ((2,2), MMA_M, MMA_N)，gmem 视图，用于写回
        acc = cute.make_rmem_tensor(thr_mma.partition_shape_C((BQ_, BQ_)), cutlass.Float32)
        acc.fill(0.0)

        # ================= s2r：ldmatrix =================
        # x4：一条指令读 4 个 8×8 子矩阵，正好是 A 的一个 16×16 块
        atom_s2r_A = cute.make_copy_atom(
            cute.nvgpu.warp.LdMatrix8x8x16bOp(transpose=False, num_matrices=4), Q.element_type)
        atom_s2r_B = cute.make_copy_atom(
            cute.nvgpu.warp.LdMatrix8x8x16bOp(transpose=False, num_matrices=4), K.element_type)
        # make_tiled_copy_A/B：按 tiled_mma 要求的 A/B 线程-值布局来配 ldmatrix
        tiled_s2r_A = cute.make_tiled_copy_A(atom_s2r_A, tiled_mma)
        tiled_s2r_B = cute.make_tiled_copy_B(atom_s2r_B, tiled_mma)
        thr_s2r_A = tiled_s2r_A.get_slice(tidx)
        thr_s2r_B = tiled_s2r_B.get_slice(tidx)
        tAsA = thr_s2r_A.partition_S(sQ)          # ldmatrix 视角下的 smem 源
        tBsB = thr_s2r_B.partition_S(sK)
        tArA = thr_s2r_A.retile(tCrA)             # 同一块寄存器，按 ldmatrix 视角重排
        tBrB = thr_s2r_B.retile(tCrB)

        # ================= GEMM：沿 K 循环 =================
        # S = Q·K^T：M=nq，N=nk，K=d。sK 是 (n, k) 且 k 连续，正好是 mma B 操作数要的布局，
        # 所以 ldmatrix 不需要 .trans。
        num_k = cute.size(tCrA, mode=[2])         # = D/16 = 4
        for k in cutlass.range_constexpr(num_k):
            cute.copy(tiled_s2r_A, tAsA[None, None, k], tArA[None, None, k])  # 1 条 x4
            cute.copy(tiled_s2r_B, tBsB[None, None, k], tBrB[None, None, k])  # 4 条 x4
            cute.gemm(tiled_mma, acc, tCrA[None, None, k], tCrB[None, None, k], acc)  # 8 条 mma

        # ================= r2g：写回 =================
        # acc 与 tCgC 形状相同，线性下标一一对应；P 的 pad 行保证不越界。
        for i in cutlass.range_constexpr(cute.size(acc)):
            tCgC[i] = acc[i].to(P.element_type)

    @cute.jit
    def __call__(self, Q: cute.Tensor, K: cute.Tensor, P: cute.Tensor):
        assert Q.element_type == cutlass.BFloat16
        B = Q.shape[0]
        Nq, Nk, Np = Q.shape[1], K.shape[1], P.shape[1]

        # 用静态 D 重建 layout：行 stride 为静态常数，编译器才能证明 16B 对齐
        src3_Q = cute.make_tensor(Q.iterator, cute.make_layout((B, Nq, self.D), stride=(Nq * self.D, self.D, 1)))
        src3_K = cute.make_tensor(K.iterator, cute.make_layout((B, Nk, self.D), stride=(Nk * self.D, self.D, 1)))  # 原来误用 Q.iterator
        dst3_P = cute.make_tensor(P.iterator, cute.make_layout((B, Np, self.D), stride=(Np * self.D, self.D, 1)))

        # ---- g2s：cp.async 16B ----
        copy_atom = cute.make_copy_atom(cute.nvgpu.cpasync.CopyG2SOp(), Q.element_type, num_bits_per_copy=128)
        tiled_copy_g2s = cute.make_tiled_copy_tv(
            copy_atom,
            cute.make_layout((self.thr_m, self.thr_n), stride=(self.thr_n, 1)),  # 线程：行×列，沿 d 相邻 → 合并访问
            cute.make_layout((1, self.VEC)))                                     # 每线程 1 行 × 8 元素

        # ---- mma：m16n8k16，bf16 输入、fp32 累加 ----
        mma_atom = cute.make_mma_atom(
            cute.nvgpu.warp.MmaF16BF16Op(cutlass.BFloat16, cutlass.Float32, (16, 8, 16)))
        # atom_layout = (WM,1,1)：warp 沿 M 排开，每 warp 独占 16 行完整的 S。
        # 每 warp 的重复次数：MMA_M = 64/(16*WM) = 1，MMA_N = 64/8 = 8，MMA_K = 64/16 = 4
        tiled_mma = cute.make_tiled_mma(mma_atom, cute.make_layout((self.WM, 1, 1)))

        self.kernel(src3_Q, src3_K, dst3_P, tiled_copy_g2s, tiled_mma).launch(
            grid=((Nq + self.BQ - 1) // self.BQ, B, 1),
            block=(self.NT, 1, 1),
            smem=2 * self.BQ * self.D * 2,   # sQ + sK，各 BQ*D 个 bf16
        )

In [5]:
# ============================================================
# test_qk_gemm.py — QK^T 分块 GEMM kernel 正确性测试
#
# 用法：把本文件内容追加到 kernel 代码所在文件的末尾（删除重复
# import），或存为独立文件后 `from qk_gemm import QK_GEMM, BQ, D`。
# 运行：python test_qk_gemm.py
# ============================================================
import torch
import cutlass
import cutlass.cute as cute
from cutlass.cute.runtime import from_dlpack

# ---- 兼容性垫片：新版 cuTe DSL 中 SmemAllocator 移到了 cutlass.memory ----
try:
    import cutlass.utils as _cu
    if not hasattr(_cu, "SmemAllocator"):
        from cutlass.memory import SmemAllocator as _SA
        _cu.SmemAllocator = _SA
except Exception:
    pass

HAS_CUDA = torch.cuda.is_available()
SENTINEL = -512.0        # bf16 可精确表示，用于检测 P 是否被写过


def _to_cute(t: torch.Tensor):
    """torch bf16 contiguous cuda tensor -> cute tensor（动态 shape）"""
    assert t.is_cuda and t.is_contiguous() and t.dtype == torch.bfloat16
    ct = from_dlpack(t.detach(), assumed_align=16)
    for meth in ("mark_layout_dynamic", "mark_internal_dynamic_layout"):
        if hasattr(ct, meth):
            ct = getattr(ct, meth)()
            break
    return ct


def _make_inputs(B, Nq, Nk, seed):
    Np = ((Nq + BQ - 1) // BQ) * BQ     # P 已 pad 成 BQ 的倍数
    g = torch.Generator(device="cuda").manual_seed(seed)
    Q = torch.randn((B, Nq, D), generator=g, device="cuda").to(torch.bfloat16)
    K = torch.randn((B, Nk, D), generator=g, device="cuda").to(torch.bfloat16)
    P = torch.full((B, Np, D), SENTINEL, device="cuda", dtype=torch.bfloat16)
    return Q, K, P, Np


def _reference(Q, K, Np):
    """逐 tile 的 fp32 参考：与 kernel 语义一一对应（越界 Q/K 行按 0）"""
    B, Nq, _ = Q.shape
    Nk = K.shape[1]
    ref = torch.zeros((B, Np, D), device=Q.device, dtype=torch.float32)
    for b in range(B):
        for t in range((Nq + BQ - 1) // BQ):
            r0 = t * BQ
            nq_t = min(r0 + BQ, Nq) - r0
            nk_t = min(r0 + BQ, Nk) - r0
            qt = torch.zeros((BQ, D), device=Q.device)
            kt = torch.zeros((BQ, D), device=Q.device)
            qt[:nq_t] = Q[b, r0:r0 + nq_t].float()
            kt[:nk_t] = K[b, r0:r0 + nk_t].float()
            ref[b, r0:r0 + BQ] = qt @ kt.T
    return ref


def _pad_checks(P, Nq, Nk, Np):
    """补零路径的强检查：pad 行 / pad 列必须精确为 0，无残留 sentinel"""
    ntiles = (Nq + BQ - 1) // BQ
    ok_row = P[:, Nq:, :].abs().max().item() == 0.0 if Np > Nq else True

    ok_col = True   # pad 列按 tile 检查：S 的列对应 tile 内 K 行偏移
    for t in range(ntiles):
        k0 = t * BQ
        if k0 >= Nk:                      # 整块 K 越界 → 该 tile 的 P 全为 0
            ok_col &= P[:, k0:k0 + BQ, :].abs().max().item() == 0.0
        elif k0 + BQ > Nk:                # 部分 K 越界 → 越界列应为 0
            v = Nk - k0
            ok_col &= P[:, k0:k0 + BQ, v:].abs().max().item() == 0.0

    leftover = (P == SENTINEL).sum().item()
    return ok_row, ok_col, leftover == 0


def run_case(name, B, Nq, Nk, use_pred=True, zero_fill=True,
             atol=0.5, rtol=0.05, seed=42):
    if not use_pred:
        # 无 predication 时 cp.async 真实越界读，必须整块对齐
        assert Nq % BQ == 0 and Nk % BQ == 0, "no-pred 用例必须整块对齐"

    Q, K, P, Np = _make_inputs(B, Nq, Nk, seed)
    fn = QK_GEMM(use_pred=use_pred, zero_fill=zero_fill, debug=False)
    cQ, cK, cP = _to_cute(Q), _to_cute(K), _to_cute(P)
    compiled = cute.compile(fn, cQ, cK, cP)
    compiled(cQ, cK, cP)
    torch.cuda.synchronize()

    ref = _reference(Q, K, Np)
    ntiles = (Nq + BQ - 1) // BQ
    errs = []

    if zero_fill:
        got, want = P.float(), ref                      # 全区比较
        errs.append((got - want).abs().max().item())
        ok = torch.allclose(got, want, atol=atol, rtol=rtol)
        r_ok, c_ok, s_ok = _pad_checks(P, Nq, Nk, Np)
        ok = ok and r_ok and c_ok and s_ok
    else:
        # 不补零：越界行列是 smem 残留垃圾，逐 tile 只比有效子块
        ok = True
        for t in range(ntiles):
            r0 = t * BQ
            nq_t = min(r0 + BQ, Nq) - r0
            nk_t = min(r0 + BQ, Nk) - r0
            g = P[:, r0:r0 + nq_t, :nk_t].float()
            w = ref[:, r0:r0 + nq_t, :nk_t]
            errs.append((g - w).abs().max().item())
            ok &= torch.allclose(g, w, atol=atol, rtol=rtol)
        r_ok = c_ok = s_ok = None

    max_err = max(errs)
    status = "PASS ✅" if ok else "FAIL ❌"
    extra = "" if r_ok is None else f"  pad行0:{r_ok} pad列0:{c_ok} 无残留:{s_ok}"
    print(f"[{status}] {name:<24s} B={B} Nq={Nq:<4d} Nk={Nk:<4d} "
          f"pred={use_pred} fill={zero_fill}  max|err|={max_err:.4g}{extra}")
    assert ok, f"测试失败: {name}"


if __name__ == "__main__":
    if not HAS_CUDA:
        raise SystemExit("需要 CUDA GPU")
    print(f"GPU: {torch.cuda.get_device_name(0)}  |  BQ={BQ}, D={D}\n")

    run_case("整块对齐",            B=2, Nq=128, Nk=128)                  # 无越界，pred/fill 空转
    run_case("非整块 Q/K",          B=3, Nq=100, Nk=70)                   # pred + 补零主路径
    run_case("极端：单行",          B=1, Nq=1,   Nk=1)                    # tile 内 63/64 行越界
    run_case("Q 多出一整块",        B=2, Nq=130, Nk=128)                  # 末块 K 全越界 → 全零块
    run_case("K 在中间块截断",      B=2, Nq=128, Nk=65)                   # 非末 tile 的部分 pad 列
    run_case("无 pred（对齐形状）", B=1, Nq=128, Nk=128, use_pred=False) # cp.async 直通路径
    run_case("pred 但不补零",       B=2, Nq=100, Nk=70, zero_fill=False)  # 只比有效区

    print("\n全部通过 ✅")


GPU: NVIDIA GeForce RTX 5070 Ti  |  BQ=64, D=64



/tmp/ipykernel_2996641/2399644835.py:42: DeprecationWarning: cutlass.utils.smem_allocator.SmemAllocator is deprecated; use cutlass.memory.SmemAllocator instead. It will be removed in a future release.
  smem = cutlass.utils.SmemAllocator()    # 警告建议改用 cutlass.memory.SmemAllocator


tQgQ: tensor<ptr<bf16, gmem, align<16>> o ((8,1),4,1):((1,0),1024,0)>
tQsQ: tensor<ptr<bf16, smem, align<16>> o ((8,1),4,1):((1,0),1024,0)>
[PASS ✅] 整块对齐                     B=2 Nq=128  Nk=128  pred=True fill=True  max|err|=0.09461  pad行0:True pad列0:True 无残留:True
tQgQ: tensor<ptr<bf16, gmem, align<16>> o ((8,1),4,1):((1,0),1024,0)>
tQsQ: tensor<ptr<bf16, smem, align<16>> o ((8,1),4,1):((1,0),1024,0)>
[PASS ✅] 非整块 Q/K                  B=3 Nq=100  Nk=70   pred=True fill=True  max|err|=0.06231  pad行0:True pad列0:True 无残留:True
tQgQ: tensor<ptr<bf16, gmem, align<16>> o ((8,1),4,1):((1,0),1024,0)>
tQsQ: tensor<ptr<bf16, smem, align<16>> o ((8,1),4,1):((1,0),1024,0)>
[PASS ✅] 极端：单行                    B=1 Nq=1    Nk=1    pred=True fill=True  max|err|=0.008387  pad行0:True pad列0:True 无残留:True
tQgQ: tensor<ptr<bf16, gmem, align<16>> o ((8,1),4,1):((1,0),1024,0)>
tQsQ: tensor<ptr<bf16, smem, align<16>> o ((8,1),4,1):((1,0),1024,0)>
[PASS ✅] Q 多出一整块                  B=2 Nq=130  Nk=128  pred=True fil